# PyCon US 2026 — Talk Brainstorm

**Title (original):** Distributing AI with Python in the Browser: Edge Inference and Flexibility Without Infrastructure
**Track:** The Future of AI with Python (Friday, Grand Ballroom A, Long Beach Convention Center)
**Track chairs:** Silona Bonewald (CitableAI), Zac Hatfield-Dodds (Anthropic)
**Format:** 30 minutes (25 min talk + 5 min Q&A)
**Experience level:** Some experience
**Speaker:** Fabio Pliger (Anaconda)

> Living document — captures decisions, references, demo ideas, and open questions from the brainstorming session for the 2026 Q2 refresh.

## 1. Original abstract (as submitted ~Nov 2025)

AI models are often complex, require a lot of resources to run and many considerations in terms of security, privacy, and performance. An emerging ecosystem of tools is making it possible to use the Browser as a great platform to run Python and consume (either by connecting or directly running) AI models. This opens the door to distributing AI to the edge without provisioning servers, managing GPUs, or handling complex infrastructure.

This talk shows how to use Python in the browser as a secure, sandboxed runtime for both edge inference or using it in agentic workflows where models are provided via an API. We will walk through an architecture where a Python runtime and models are delivered as assets, with all inference happening in different ways in (or through) the user's browser. Along the way, we will look at how this fits into the broader AI‑in‑browser ecosystem: interoperating with existing JavaScript model runners, leveraging WebGPU/WebNN for acceleration, mixing Python‑based preprocessing with client‑side inference engines or simply a gateway to AI services like OpenAI or Anthropic.

Security and privacy are first‑class concerns: the browser sandbox limits what code can do, and keeping inference and related compute on the client means user data does not need to leave the device. We will also be honest about the trade‑offs: bundle sizes, model constraints, performance, and when a hybrid edge/cloud approach makes more sense than going fully client‑side. Attendees will leave with concrete patterns, example architectures, and a clear path to experiment with Python‑powered AI at the edge, no special infrastructure required.

## 2. Direction decisions

- **Centerpiece:** an **agent** running in the browser, not a bare inference demo. Python-in-browser is the orchestration layer; the model can be local, remote, or a mix.
- **Title stays as submitted.** "Distributing AI with Python in the Browser: Edge Inference and Flexibility Without Infrastructure." Don't pitch a rename.
- **PyScript framing.** PyScript is **one of the choices** the talk presents — the runtime used in the live demo because the audience is Python-native — but **PyScript is not the goal of the talk**. Patterns and architecture generalize across browser-Python runtimes (Pyodide directly, MicroPython for PyScript, even non-PyScript embeddings).
- **Architecture style:** **hybrid (edge + cloud)** as the headline pattern. A fully-local example is a *nice-to-have / standalone bonus*, not the spine of the talk.
- **MCP coverage = (b) mini-primer.** ~2 minutes inside the architecture section. What MCP is, why it exists, why a Python-in-browser runtime is a natural MCP host/client. Audience leaves with a working understanding.
- **Anatomy of an agent = brief educational baseline.** Show the four parts in general, then map them onto the browser (same concepts, different technical details). Single slide, ~2 min.
- **Anaconda framing.** Speaker intro will mention the Anaconda/PyScript affiliation. The talk itself doesn't open with it — credibility is earned through a working demo, then the affiliation lands organically.
- **Privacy/sandbox angle:** keep this as a first-class theme but lean on it more strongly than the original abstract did, given the 2026 regulatory environment.
- **Honesty:** keep the trade-offs slide. Bundle sizes, cold starts, model ceiling, when *not* to do this.
- **Demo strategy:** build several candidate demos in a separate spike session, pick the headline closer to the talk. Starting with §8 #4 (local-first router).
- **Working folder:** `pyconUS_2026/`. Will be merged into a personal talks repo later.
- **Spike session:** demo prototyping (PyScript + MCP SDK + WebLLM/Transformers.js JS interop + SSE streaming) is its own dedicated session, not this one.

### Still open (for after the spike)
- Final headline demo pick (drives slide design).
- Backup demo recordings for conference-Wi-Fi resilience.
- Repo polish for the takeaway slide.

## 3. Narrative through-line — draft

The single sentence the audience should be able to repeat after the talk:

> **"In 2026, the browser is a real platform for AI agents — and Python is the right language to glue one together."**

The talk has three movements that build to that line:

1. **Three things shifted in the last twelve months.** WebGPU shipped in every major browser (Nov 2025). Small models grew up — Phi-4-mini, Gemma 4 do real tool calling. MCP became the de-facto agent-tools standard. Each of these alone wouldn't move the needle; together they unlock something new.
2. **An agent is just three parts** — model, tools, loop — and the browser is, surprisingly, a good place to keep all three. Python is great at the loop and the tools. The model can run wherever it runs fastest.
3. **Here's how that looks in practice** — live demo, architecture, honest trade-offs.

### Notes
- The line is not "PyScript runs Python in the browser" — that's *how*, not *why*.
- The line is not "MCP is the future" — that's a piece, not the message.
- The line *is* about distribution: agents that ship as static assets, run on the user's device, respect their data, and don't need infrastructure.
- The original abstract's last sentence ("a clear path to experiment with Python-powered AI at the edge, no special infrastructure required") is still the right closing energy. Land on it.

## 4. The 2026 Q2 landscape — facts to anchor the talk

Highlights from a search pass on May 5, 2026. These are the data points worth weaving into the "why now" framing.

### Browser Python runtime
- **PyScript 2026.2.1** is current; bundled **Pyodide 0.29.3**.
- MicroPython runtime variant at ~303 KB, <100 ms cold start — relevant for "Python ships as part of the page" framing.

### WebGPU
- **Shipped by default** across Chrome, Firefox, Edge, and Safari on **Nov 25, 2025** — first time all four majors support it. ~82.7% global coverage.
- Real numbers worth quoting:
  - Llama 3.1 8B (4-bit): ~41 tok/s in browser.
  - Phi-3.5 mini: ~71 tok/s in browser; ~90 tok/s on M3.
  - Phi-3 mini token latency dropped 320 ms → 85 ms (WebGL → WebGPU) on M2 Air — 3.8×.
  - 3–5× over WebGL, 10–15× over plain WebAssembly for transformer workloads.

### WebNN
- W3C Candidate Recommendation reached **Jan 2026**.
- Chrome/Edge have working impls; Safari and Firefox **have not shipped** WebNN as of Apr 2026.
- Realistic "default path" timeline: **2027**. Mention as a near-future trajectory, not as something to rely on today.

### Browser inference engines
- **Transformers.js v4** (announced at Web AI Summit 2025): 53% smaller bundles, build time 2 s → 200 ms. Recent model adds: Qwen3 Next, Apple OpenELM, ModernBERT.
- **WebLLM** (`@mlc-ai/web-llm`) typically wins on raw LLM throughput vs. Transformers.js.
- Both production-ready for SLMs ≤ ~3B on consumer hardware.

### Small models that actually do tool calling
- **Phi-4-mini** (Microsoft): function calling now supported.
- **Gemma 4 9B** (Apr 2, 2026): built-in tool calling, recommended for local agents.
- **Gemma 4 E2B/E4B**: 5 GB RAM at 4-bit, 128K context, native audio input, Apache 2.0.
- **Qwen 3.5 Small**: 0.8B–9B, 256K context, 201 languages.
- The story is: **local SLMs are now genuinely good enough to drive an agent loop** — this is new since the abstract was submitted.

### MCP
- 78% of enterprise AI teams have at least one MCP-backed agent in production (Apr 2026 survey).
- 67% of CTOs name MCP their default integration standard.
- Public MCP server registry: 1,200 (Q1 2025) → 9,400+ (Apr 2026); +18% MoM in Q1 2026.
- Anthropic reports ~97M monthly SDK downloads (Mar 2026).
- Browser-relevant: OAuth 2.1 + PKCE flows for browser-based agents on Anthropic's 2026 roadmap; first-class TS SDK.
- Supported clients: Claude (native), ChatGPT (Apps SDK + Connectors), Gemini (Mar 2026), Cursor, Windsurf, Zed, JetBrains AI, Vercel AI SDK, OpenAI Agents SDK.

### Agentic browser landscape (context, not necessarily on-stage)
- AI browser market: $4.5B (2024) → $76.8B (2034) projected, 32.8% CAGR.
- Agentic traffic up **7,851% YoY** 2025→2026.
- Tools: browser-use, Vercel agent-browser, Stagehand v3 (Feb 2026, AI-native rewrite, 44% faster).
- Useful as a "where the puck is going" callout, even if our talk is *Python-runs-in-the-browser* rather than *agent-controls-a-browser*.

## 5. Anatomy of an agent — draft for the educational segment

Target: **single slide, ~2 minutes of stage time.** Not a teaching block; a shared mental model so the architecture section lands.

The slide shows the same four parts twice — once in general, once in the browser — to make the point that **the architecture is invariant; only the implementation details change**.

### The four parts (general)
1. **Model** — the part that *decides what to do next*. Anywhere it can run.
2. **Tools** — functions the model is allowed to call. Increasingly described via **MCP**.
3. **Loop** — the runtime that drives execution: ask the model, run the tool, feed the result back, repeat until done.
4. **Context / memory** — what the model sees on each iteration: instructions, tool results, history.

### The same four parts (in the browser)

| Part | In general | In the browser |
|---|---|---|
| Model | Anywhere — laptop, server, API | Local SLM via WebGPU **or** remote API via `fetch`. Same role. |
| Tools | Functions / APIs | MCP servers (HTTP) + Python-native tools (Pandas, etc.) + DOM access. Same role. |
| Loop | Wherever you write code | Python in PyScript, in the user's tab. Same role. |
| Context | What the model sees | Same, plus: page DOM, sandboxed user files, browser-local memory. |

**The teaching beat:** *"The architecture doesn't change in the browser. Only the implementation details — bundle size, cold start, what each part can talk to — change. That's the whole talk in one slide."*

### The bridge sentence into the architecture
> "An agent is a model + tools + a loop. Python is great at writing loops and orchestrating tools. The browser is a great place to keep user data. Put them together and you get an agent that ships as static assets and runs on the user's device."

### Why this framing for *this* talk
- Maps cleanly onto the architecture diagram in §9: model = local SLM / remote API, tools = MCP servers + Python-native helpers, loop = PyScript code.
- Lets the speaker introduce MCP naturally inside row (2) without a detour.
- Makes the case that the *browser doesn't change what an agent is* — only where the parts run and what constraints they have.

### What to *not* cover here
- ReAct / planner-executor / multi-agent patterns (mention exists; don't teach).
- RAG, embeddings, vector stores (out of scope unless the demo uses them).
- Eval / observability / safety — could be a closing nod, not a section.

## 6. Demo implementation: PyScript as the chosen runtime

> The talk presents browser-Python as a category. The **demo** uses PyScript because we're at PyCon and the audience can read it. The patterns generalize.

The agent code shown on stage is Python, running in PyScript, in the audience's browser tab. JS is allowed but minimized — when JS appears, it's because something genuinely *should* be JS (the GPU model runtime).

### What's Python (PyScript) on stage
- The **agent loop** — the main `while not done:` driving the agent.
- The **tool implementations** that don't need DOM/JS — Pandas analysis, validation, formatting, parsing, lightweight planning.
- The **MCP client** — Python SDK over `fetch` (via JS interop). The MCP Python SDK is the obvious dependency to lean on.
- The **router / policy** — "should this go local or remote?" lives in Python.
- The **model gateway abstraction** — a thin Python class with a single `.complete()` / `.stream()` interface; concrete implementations call either WebLLM (JS) or a remote API via `fetch`.

### What stays JS
- The actual **WebGPU model runtime** (WebLLM / Transformers.js). Don't reinvent these in Python; call them via PyScript's JS interop. This is honest and important to show.
- DOM rendering glue if it's awkward in Python.

### Why this implementation choice for *this* talk
- Reinforces the talk's core message: Python is the orchestration layer; the model is whatever runs fastest.
- Gives the audience code they could conceivably read on stage.
- Avoids the trap where the headline demo is secretly 80% TypeScript.
- Lets the architecture diagram say "Python runtime" generically and the demo say "PyScript specifically" — the abstraction holds.

### Spike session prerequisites (separate session — see §11)
- Confirm MCP Python SDK works under Pyodide (pure-Python deps fine; native deps are the risk).
- Confirm streaming responses (SSE) work cleanly through Pyodide's `fetch` shim.
- Confirm WebLLM/Transformers.js JS APIs called via the `js` module behave under load.
- Each is roughly a 1-hour investigation; all three together = the spike's first phase.

## 7. Working outline (25 min + 5 min Q&A) — draft v2

| # | Section | Minutes | Notes |
|---|---|---|---|
| 1 | **Hook** — live PyScript agent doing something non-trivial in a browser tab | 3 | Built in PyScript, runs offline-friendly. Recorded fallback ready. |
| 2 | **Why now (2026)** — WebGPU shipped in all 4 browsers, SLMs do tool calling, MCP is a standard | 2 | Use 3–4 numbers from §4, not all of them. |
| 3 | **Anatomy of an agent** — model + tools + loop (+ context). Single slide, brief | 2 | See §5. Just enough vocabulary for the rest. |
| 4 | **The architecture** — Python runtime as glue: tool dispatch, MCP client, model gateway, optional local fallback | 6 | Headline diagram + walk through the demo's actual PyScript code. |
| 5 | **MCP mini-primer** — what MCP is, why it exists, why a Python-in-browser runtime is a natural MCP client | 2 | Lives inside §4 narratively, but worth budgeting separately. |
| 6 | **Three reference patterns** — pure-edge, gateway-only, hybrid router | 3 | Quick taxonomy. Hybrid router = the demo's pattern. |
| 7 | **Honest trade-offs** — bundle size, cold start, model ceiling, when *not* to do this | 4 | Audiences remember the speaker who told them when *not* to use the thing. |
| 8 | **Takeaways + repo + what's next** | 2 | One CTA. "Clone this, you have an agent in 5 minutes." |
| — | **Q&A** | 5 | |

**Total:** 24 minutes content + 5 min Q&A. 1 minute of slack baked in.

### Risks
- §3 + §4 + §5 = 10 minutes of architecture density. Rehearse strict; this is where the talk overruns.
- The hook demo *must* survive conference Wi-Fi. Cache aggressively; have a recorded fallback that's indistinguishable from live.
- §5 (MCP primer) is the highest risk of "speaker goes 90 seconds long because they love the topic." Time it.

## 8. Demo / example ideas

The demo carries the talk. Need *one* primary demo and optionally a "look, also fully-local" coda. Built in a separate spike session.

### Criteria
- Visibly does real work (not a chatbot reply).
- Showcases the **agent loop** (model → tool → result → model).
- Touches the **hybrid** architecture: at least one local thing and at least one remote tool/model call.
- Survives bad conference Wi-Fi (cache aggressively; have a recorded fallback).
- Fits in ~2–3 minutes of stage time.
- Code is **PyScript** (the audience can read it on screen).

### Candidate demos
1. **Personal data analyst (private CSV).** User drops a CSV (sales, health, expenses — pick benign), agent answers questions. Pandas-in-Pyodide runs the analysis locally; remote LLM does the reasoning/planning; data never leaves the tab. Strong privacy story.
2. **Local research assistant.** Page-aware agent that reads the current tab, summarizes, extracts entities, calls one MCP tool (e.g., a remote search or a calendar). Shows the agent loop crisply.
3. **Notebook copilot in the browser.** Pyodide kernel + small local SLM as a fallback "offline mode," remote model when online. Speaks to the PyCon audience directly.
4. **Local-first router (starting candidate).** A tiny local model (intent / safety / language detection) routes each request to either a local Phi-4-mini call or a remote frontier model. Cleanest illustration of the hybrid pattern. **Probably the strongest fit for the headline demo.**

### Router demo — proposed on-stage flow (3 prompts, 3 paths)
The router decision is overlaid on the UI so the audience *sees* which path was taken.

| # | Prompt | Routing decision | What it demonstrates |
|---|---|---|---|
| 1 | "Convert this date to ISO format: May 5, 2026" | Local SLM, no tool call | Simple, private, fast — no need to leave the tab. |
| 2 | "Summarize this 200-line CSV I just dropped in" | Local Pandas tool + remote model frames the answer | Hybrid: data stays local, model frames it. |
| 3 | "Find the top 3 climate stories last week and put them in my notes" | Remote frontier model → calls `web_search` MCP tool → reads results → calls `save_to_file` tool → done | Full agent loop on the remote-routing path: model → tool → result → model → tool → done. |

*Drafting note:* prompt #3 was originally framed as a no-tool-call remote query, which doesn't really demonstrate the agent loop. The version above is the corrected one — full multi-step loop, still routed to the remote model. Worth revisiting after the spike if a stronger third example emerges.

### Standalone "fully local" bonus
- Phi-4-mini or Gemma 4 E2B doing tool-calling on WebGPU, no network. Useful as a 30-second cameo to prove the local path is real, then move on.

### Open: which one becomes the headline?
Decision deferred to after the spike. Starting candidate: #4.

## 9. Slide concepts — for the design pass

Captured here so the spike session and the slide pass have a starting point. Not yet committed.

### The architecture diagram (the talk's anchor slide)
**Center:** a labeled box "Python runtime in the browser (PyScript)."
**Four arrows out:**
- ⬆ to **Local model** (WebGPU runtime — WebLLM / Transformers.js)
- ➡ to **Remote model** (Anthropic / OpenAI / self-hosted) via `fetch`
- ⬇ to **MCP servers** (tools — local stdio bridge or remote HTTP)
- ⬅ to **Python-native tools** (Pandas, validation, parsing — same process)

**One loop arrow** wrapping the center to convey "the agent loop runs here."
**Caption:** *"Python decides; the model and tools run wherever's best."*

This single slide, returned to multiple times, is what the talk stands or falls on visually.

### The anatomy slide (single slide, ~2 min)
Two rows, four columns. Same four parts, twice.

Row 1 — *In general*: Model · Tools · Loop · Context
Row 2 — *In the browser*: Local SLM / Remote API · MCP + Python-native + DOM · PyScript loop · DOM + sandboxed files + memory
With an arrow wrapping back to the first column.
Bottom line: *"Same architecture. Different implementation details."*

### The "why now" slide
Three rows. One number per row.
- WebGPU: shipped in **all four** major browsers (Nov 2025)
- SLMs: **Phi-4-mini, Gemma 4 9B** — real function calling, run on a laptop
- MCP: **97M monthly SDK downloads**, 9,400+ public servers
Caption: *"These three didn't matter alone. Together they do."*

### The trade-offs slide
Two columns. "Works well for" vs. "Don't use this for."

| Works well for | Don't use this for |
|---|---|
| Privacy-sensitive workloads | Multi-GB frontier models |
| Offline / intermittent connectivity | Workloads needing low cold-start under 200ms |
| Personal tools, single-user agents | Multi-tenant high-throughput backends |
| Demos, prototypes, MVPs | Anything with hard model-quality SLAs |
| Anything where infra cost is prohibitive | Workloads where the user's device is constrained |

*Confirm rows after the spike — real demo work usually surfaces constraints we hadn't anticipated.*

### The CTA slide
- Title: "Try it"
- One QR code → the repo
- Three bullets: "Clone. Run a server. Open a tab." (or similar — the actual three steps)
- "Questions?"

## 10. Status & remaining questions

### Resolved this session
- ✅ MCP depth = (b) mini-primer, ~2 min, inside the architecture section.
- ✅ Architecture style = hybrid as headline; fully-local as standalone bonus.
- ✅ Agent built in PyScript (Python audience).
- ✅ PyScript is one of the choices, not the goal of the talk.
- ✅ Title stays as submitted.
- ✅ Anatomy of an agent = brief educational baseline. Show four parts in general, then in the browser. Single slide, ~2 min.
- ✅ Anaconda framing = speaker intro handles it; demo earns credibility.
- ✅ Headline demo strategy = build several, pick later. Starting with §8 #4 (local-first router).
- ✅ Router demo prompts = 3 prompts hitting 3 routing paths (see §8). Original example #3 dropped, web-search agent locked in.
- ✅ Working folder = `pyconUS_2026/`.
- ✅ Spike session is separate from this brainstorm.

### Open — for after the spike
1. Final headline demo pick.
2. Architecture diagram visual (concept in §9 — needs design pass).
3. Anatomy slide visual (concept in §9 — needs design pass).
4. Trade-offs slide rows (table in §9 — confirm after demo work surfaces real constraints).
5. Repo for takeaway slide — does the demo code become the public repo, or do we curate?
6. Backup demo recordings for offline-Wi-Fi resilience.

## 11. Spike session handoff

This brainstorm session is closing. The next session is dedicated to the demo spike. Start there with this notebook as context, plus the items below.

### Goal of the spike
Build enough of the **router demo (§8 #4)** to validate three things:
1. The architecture works end-to-end in PyScript.
2. The audience-facing flow (3 prompts / 3 paths) reads clearly on stage.
3. Conference-grade resilience — caching, recorded fallback, offline behavior — is achievable.

If any of those fall over, that's the spike's most valuable output: telling us early what to redesign.

### Technical risks to validate first (~1 hour each, in this order)
1. **MCP Python SDK under Pyodide.** Pure-Python deps fine; native deps are the risk. Confirm a basic `list_tools` + `call_tool` round-trip against an HTTP MCP server.
2. **Streaming responses (SSE) through Pyodide's `fetch` shim.** Confirm token streaming from Anthropic / OpenAI APIs works without buffering surprises.
3. **WebLLM or Transformers.js called from PyScript via JS interop.** Confirm a small model loads, generates, and round-trip is fast enough to be live-demo-able.

If all three pass: build the router demo. If any fail: redesign that part of the architecture before going further.

### What "done" looks like for the spike
- Working router demo with the 3 prompts hitting the 3 paths.
- Recorded fallback video that's visually indistinguishable from live.
- Bundle size measured and recorded.
- Cold start time measured and recorded.
- Notes on what was harder than expected — feeds the trade-offs slide (§9).

### Optional, if time permits during the spike
- Build one of demo candidates 1, 2, or 3 (§8) to give us alternates if the router doesn't land well in rehearsal.
- Sketch the architecture diagram in code-shaped form (Mermaid is fine) to anchor the slide design.

### Out of scope for the spike
- Slide design (separate later session).
- Speaker rehearsal / timing (separate later session).
- Any infrastructure beyond `python -m http.server`.

## 12. References

### PyScript / Pyodide
- [PyScript docs (2026.1.1)](https://docs.pyscript.net/2026.1.1/user-guide/what/)
- [PyScript releases](https://github.com/pyscript/pyscript/releases)
- [Pyodide project](https://pyodide.org/)
- [Pyodide GitHub](https://github.com/pyodide/pyodide)
- [Anaconda blog — PyScript Updates: Bytecode Alliance, Pyodide, MicroPython](https://www.anaconda.com/blog/pyscript-updates-bytecode-alliance-pyodide-and-micropython)
- [Beyond the Server: Python in the Browser with Pyodide and WebAssembly (2026 Guide)](https://glinteco.com/en/post/beyond-the-server-running-high-performance-python-in-the-browser-with-pyodide-and-webassembly-2026-guide/)

### MCP
- [MCP 2026 Roadmap (official blog)](https://blog.modelcontextprotocol.io/posts/2026-mcp-roadmap/)
- [MCP roadmap (modelcontextprotocol.io)](https://modelcontextprotocol.io/development/roadmap)
- [MCP Adoption Statistics 2026](https://www.digitalapplied.com/blog/mcp-adoption-statistics-2026-model-context-protocol)
- [MCP Hits 97M Downloads](https://www.digitalapplied.com/blog/mcp-97-million-downloads-model-context-protocol-mainstream)
- [MCP's biggest growing pains for production use will soon be solved (The New Stack)](https://thenewstack.io/model-context-protocol-roadmap-2026/)
- [Model Context Protocol — Wikipedia](https://en.wikipedia.org/wiki/Model_Context_Protocol)

### WebGPU / WebNN
- [WebGPU 2026: 70% Browser Support, 15× Performance Gains](https://byteiota.com/webgpu-2026-70-browser-support-15x-performance-gains/)
- [WebGPU Browser AI Inference (2026 cost analysis)](https://www.buildmvpfast.com/blog/webgpu-browser-ai-inference-cost-savings-2026)
- [WebAssembly for LLM Inference (ONNX + WebGPU, Jan 2026)](https://dasroot.net/posts/2026/01/webassembly-llm-inference-browsers-onnx-webgpu/)
- [WebLLM paper (arXiv 2412.15803v2)](https://arxiv.org/html/2412.15803v2)
- [Web Platform Status — WebNN](https://webstatus.dev/features/webnn)
- [W3C Web Neural Network API (TR)](https://www.w3.org/TR/webnn/)
- [WebNN Overview — Microsoft Learn](https://learn.microsoft.com/en-us/windows/ai/directml/webnn-overview)

### Browser inference engines
- [Transformers.js docs — Hugging Face](https://huggingface.co/docs/transformers.js/en/index)
- [Transformers.js GitHub](https://github.com/huggingface/transformers.js/)
- [Transformers.js vs ONNX Runtime Web 2026](https://www.pkgpulse.com/guides/transformersjs-vs-onnx-runtime-web-2026)
- [Build a Private, Local AI Browser Assistant with Transformers.js (Apr 2026)](https://mindwiredai.com/2026/04/28/build-a-private-local-ai-browser-assistant-with-transformers-js/)
- [Intel: A Guide to In-Browser LLMs](https://www.intel.com/content/www/us/en/developer/articles/technical/web-developers-guide-to-in-browser-llms.html)

### Small language models with tool calling
- [Best Open-Source SLMs in 2026 (BentoML)](https://www.bentoml.com/blog/the-best-open-source-small-language-models)
- [Top 15 SLMs for 2026 — DataCamp](https://www.datacamp.com/blog/top-small-language-models)
- [Phi-4 Mini vs. Gemma 3 vs. Qwen 2.5 (Botmonster)](https://botmonster.com/posts/phi-4-mini-vs-gemma-3-vs-qwen-25-best-slm-coding-2026/)
- [Best Small AI Models to Run with Ollama (2026)](https://localaimaster.com/blog/small-language-models-guide-2026)

### Agentic browsers (context, not core)
- [The State of AI & Browser Automation in 2026 — Browserless](https://www.browserless.io/blog/state-of-ai-browser-automation-2026)
- [11 Best AI Browser Agents in 2026](https://www.firecrawl.dev/blog/best-browser-agents)
- [Cloudflare Agents Week 2026 recap](https://blog.cloudflare.com/agents-week-in-review/)

### PyCon US 2026
- [The Future of AI with Python — track page](https://us.pycon.org/2026/tracks/ai/)
- [Talks schedule](https://us.pycon.org/2026/schedule/talks/)
- [Simon Willison on PyCon US 2026 (new AI + security tracks)](https://simonwillison.net/2026/Apr/17/pycon-us-2026/)